# 21 — Plan B Final Interactive Interface

Notebook 21 is the user-facing layer of Plan B.

It does not train a model and does not change the Plan B decision engine.

It packages the validated Notebook 18 reference library into a standalone HTML interface where the user can enter:

- latitude
- longitude
- bridge length
- bridge width
- optional DTV
- optional Baustoff

The interface calculates the same transparent similarity + performance decision logic established in Notebook 20 and displays:

- recommended Bauwerksart;
- ranked alternative Bauwerksarten;
- reference bridges supporting the recommendation;
- geographic map of the scenario and reference bridges;
- active decision inputs and weights.

This notebook is the presentation/interface layer only.

## 01 — Project paths

Only the agreed project roots are used:

```text
C:\Datenanalyse\final Project\Dataset_PlanA-B
C:\Datenanalyse\final Project\Output_PlanA-B
```

Input:

```text
Output_PlanA-B/
└── 18_Plan_B_Reference_Library/
    └── plan_b_reference_library.parquet
```

Notebook 20 is used as the validated logic reference.

Output:

```text
Output_PlanA-B/
└── 21_Plan_B_Final_Interactive_Interface/
    ├── plan_b_final_interface.html
    ├── plan_b_interface_data.json
    └── 21_plan_b_interface_manifest.json
```

In [6]:
from pathlib import Path
import json
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Datenanalyse\final Project")
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"

LIBRARY_INPUT = (
    OUTPUT_ROOT
    / "18_Plan_B_Reference_Library"
    / "plan_b_reference_library.parquet"
)

SOURCE_20 = (
    OUTPUT_ROOT
    / "20_Plan_B_User_Scenario"
    / "plan_b_scenario_result.csv"
)

OUTPUT_DIR = OUTPUT_ROOT / "21_Plan_B_Final_Interactive_Interface"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HTML_OUTPUT = OUTPUT_DIR / "plan_b_final_interface.html"
DATA_JSON = OUTPUT_DIR / "plan_b_interface_data.json"
MANIFEST_JSON = OUTPUT_DIR / "21_plan_b_interface_manifest.json"

if not LIBRARY_INPUT.exists():
    raise FileNotFoundError(f"Notebook 18 library not found:\n{LIBRARY_INPUT}")

reference = pd.read_parquet(LIBRARY_INPUT)

if len(reference) != 52214:
    raise ValueError(f"Expected 52,214 reference bridges; found {len(reference)}")

if not reference["bridge_id"].is_unique:
    raise ValueError("bridge_id must be unique.")

print("[PASS] reference library loaded")
print("Reference rows:", len(reference))

[PASS] reference library loaded
Reference rows: 52214


## 02 — Interface data contract

Only the fields needed by the interactive Plan B engine are exported to the browser.

The interface does not expose the full 86-predictor ML contract because Plan B similarity and decision logic operate above that frozen condition model.

In [7]:
required_fields = [
    "bridge_id",
    "bauwerksart_text",
    "baustoffklasse",
    "latitude",
    "longitude",
    "laenge",
    "breite",
    "dtv_reference",
    "performance_score",
]

missing = [c for c in required_fields if c not in reference.columns]

if missing:
    raise ValueError(
        "Reference library is missing interface fields:\n"
        + "\n".join(missing)
    )

map_data = reference[required_fields].copy()

numeric_fields = [
    "latitude",
    "longitude",
    "laenge",
    "breite",
    "dtv_reference",
    "performance_score",
]

for c in numeric_fields:
    map_data[c] = pd.to_numeric(map_data[c], errors="coerce")

map_data = map_data[
    map_data["latitude"].between(-90, 90, inclusive="both")
    & map_data["longitude"].between(-180, 180, inclusive="both")
].copy()

map_data = map_data.replace({np.nan: None})

print("Browser reference records:", len(map_data))
print("[PASS] interface data contract")

Browser reference records: 51428
[PASS] interface data contract


In [8]:
payload = {
    "reference_count": int(len(map_data)),
    "references": map_data.to_dict(orient="records"),
    "base_weights": {
        "bauwerksart": 0.30,
        "baustoff": 0.20,
        "length": 0.15,
        "width": 0.10,
        "dtv": 0.15,
        "distance": 0.10,
    },
    "max_distance_km": 50.0,
    "top_references_per_type": 10,
    "min_references_per_type": 5,
}

DATA_JSON.write_text(
    json.dumps(payload, ensure_ascii=False, separators=(",", ":")),
    encoding="utf-8",
)

print("[PASS] browser data written")
print("JSON size MB:", round(DATA_JSON.stat().st_size / 1024**2, 2))

[PASS] browser data written
JSON size MB: 12.61


## 03 — Build standalone interactive interface

The generated HTML:

- contains the Plan B form;
- calculates the scenario recommendation in the browser;
- uses the same weighting structure as Notebook 20;
- shows a Leaflet map;
- marks the proposed bridge;
- marks supporting reference bridges;
- displays ranked Bauwerksart alternatives.

No model training occurs in the browser.

In [9]:
data_json = DATA_JSON.read_text(encoding="utf-8")

html = r'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Plan B — Bridge Type Recommendation</title>

<link
 rel="stylesheet"
 href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"
/>

<style>
body {
    margin: 0;
    font-family: Arial, sans-serif;
    background: #f5f5f5;
    color: #222;
}
header {
    padding: 18px 24px;
    background: #222;
    color: white;
}
main {
    display: grid;
    grid-template-columns: 360px 1fr;
    gap: 16px;
    padding: 16px;
}
.panel {
    background: white;
    border: 1px solid #ccc;
    border-radius: 8px;
    padding: 16px;
}
label {
    display: block;
    margin-top: 10px;
    font-weight: 600;
}
input, select, button {
    width: 100%;
    box-sizing: border-box;
    padding: 9px;
    margin-top: 5px;
    border: 1px solid #aaa;
    border-radius: 5px;
}
button {
    margin-top: 16px;
    cursor: pointer;
    font-weight: 700;
}
#map {
    height: 650px;
    border-radius: 8px;
}
.result {
    margin-top: 16px;
    padding: 14px;
    border: 1px solid #aaa;
    border-radius: 6px;
}
.recommendation {
    font-size: 22px;
    font-weight: 700;
}
table {
    width: 100%;
    border-collapse: collapse;
    margin-top: 12px;
}
th, td {
    border: 1px solid #ccc;
    padding: 6px;
    text-align: left;
    font-size: 13px;
}
th {
    background: #eee;
}
.note {
    font-size: 12px;
    color: #555;
    margin-top: 12px;
}
@media (max-width: 900px) {
    main { grid-template-columns: 1fr; }
    #map { height: 500px; }
}
</style>
</head>

<body>

<header>
<h1>Plan B — Bridge Type Recommendation</h1>
<div>Reference-based decision support for a proposed bridge scenario</div>
</header>

<main>

<section class="panel">

<h2>Proposed Bridge</h2>

<label>Latitude</label>
<input id="lat" type="number" step="any" value="50.7374">

<label>Longitude</label>
<input id="lon" type="number" step="any" value="7.0982">

<label>Length (m)</label>
<input id="length" type="number" step="any" value="80">

<label>Width (m)</label>
<input id="width" type="number" step="any" value="12">

<label>DTV (optional)</label>
<input id="dtv" type="number" step="any" placeholder="optional">

<label>Baustoff (optional)</label>
<input id="material" type="text" placeholder="e.g. Stahlbeton">

<button onclick="runRecommendation()">
Calculate recommendation
</button>

<div id="result" class="result">
Enter the scenario and calculate the recommendation.
</div>

<div class="note">
The result is a planning decision-support output based on comparable existing
bridges. It is not structural design, FEM, dimensioning, code checking or
formal approval.
</div>

</section>

<section class="panel">
<div id="map"></div>
</section>

</main>

<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>

<script>
const DATA = DATA_PLACEHOLDER;

const BASE_WEIGHTS = DATA.base_weights;
const MAX_DISTANCE_KM = DATA.max_distance_km;
const TOP_REFERENCES = DATA.top_references_per_type;
const MIN_REFERENCES = DATA.min_references_per_type;

const map = L.map('map').setView([51.1657, 10.4515], 6);

L.tileLayer(
    'https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
    {
        attribution: '&copy; OpenStreetMap contributors'
    }
).addTo(map);

let scenarioMarker = null;
let referenceMarkers = [];

function haversineKm(lat, lon, rlat, rlon) {
    const R = 6371.0088;
    const p1 = lat * Math.PI / 180;
    const p2 = rlat * Math.PI / 180;
    const dp = (rlat - lat) * Math.PI / 180;
    const dl = (rlon - lon) * Math.PI / 180;

    const a =
        Math.sin(dp / 2) ** 2 +
        Math.cos(p1) * Math.cos(p2) * Math.sin(dl / 2) ** 2;

    return R * 2 * Math.asin(Math.sqrt(Math.min(1, a)));
}

function numericSimilarity(x, value, scale) {
    if (x === null || x === undefined || !Number.isFinite(Number(x))) {
        return null;
    }
    return Math.exp(-Math.abs(Number(x) - value) / scale);
}

function dtvSimilarity(x, value) {
    if (x === null || x === undefined || !Number.isFinite(Number(x))) {
        return null;
    }

    const denominator = Math.max(
        Math.max(Math.abs(Number(x)), Math.abs(value)),
        1
    );

    return Math.max(
        0,
        Math.min(
            1,
            1 - Math.abs(Number(x) - value) / denominator
        )
    );
}

function clearReferenceMarkers() {
    referenceMarkers.forEach(m => map.removeLayer(m));
    referenceMarkers = [];
}

function runRecommendation() {

    const lat = Number(document.getElementById('lat').value);
    const lon = Number(document.getElementById('lon').value);
    const length = Number(document.getElementById('length').value);
    const width = Number(document.getElementById('width').value);

    const dtvText = document.getElementById('dtv').value.trim();
    const material = document.getElementById('material').value.trim();

    const dtv = dtvText === "" ? null : Number(dtvText);
    const baustoff = material === "" ? null : material;

    if (
        !Number.isFinite(lat) ||
        !Number.isFinite(lon) ||
        !Number.isFinite(length) ||
        !Number.isFinite(width) ||
        lat < -90 || lat > 90 ||
        lon < -180 || lon > 180 ||
        length <= 0 ||
        width <= 0
    ) {
        alert("Please enter valid latitude, longitude, length and width.");
        return;
    }

    if (dtv !== null && (!Number.isFinite(dtv) || dtv < 0)) {
        alert("DTV must be empty or a non-negative number.");
        return;
    }

    const refs = DATA.references;

    const lengthValues = refs
        .map(r => Number(r.laenge))
        .filter(Number.isFinite)
        .sort((a,b) => a-b);

    const widthValues = refs
        .map(r => Number(r.breite))
        .filter(Number.isFinite)
        .sort((a,b) => a-b);

    const q75 = arr => arr[Math.floor(0.75 * (arr.length - 1))];

    const lengthScale = Math.max(q75(lengthValues), 1);
    const widthScale = Math.max(q75(widthValues), 1);

    const active = {
        length: true,
        width: true,
        distance: true,
        dtv: dtv !== null,
        baustoff: baustoff !== null
    };

    const raw = {};

    Object.keys(active).forEach(k => {
        if (active[k]) raw[k] = BASE_WEIGHTS[k];
    });

    const sum = Object.values(raw).reduce((a,b) => a+b, 0);

    const weights = {};
    Object.keys(raw).forEach(k => {
        weights[k] = raw[k] / sum;
    });

    const scored = [];

    refs.forEach(r => {

        const rlat = Number(r.latitude);
        const rlon = Number(r.longitude);

        if (!Number.isFinite(rlat) || !Number.isFinite(rlon)) return;

        const distance = haversineKm(lat, lon, rlat, rlon);

        if (distance > MAX_DISTANCE_KM) return;

        const components = {
            length: numericSimilarity(
                r.laenge, length, lengthScale
            ),
            width: numericSimilarity(
                r.breite, width, widthScale
            ),
            distance: Math.exp(-distance / 50)
        };

        if (dtv !== null) {
            components.dtv = dtvSimilarity(r.dtv_reference, dtv);
        }

        if (baustoff !== null) {
            components.baustoff =
                r.baustoffklasse !== null &&
                String(r.baustoffklasse).toLowerCase()
                === baustoff.toLowerCase()
                    ? 1
                    : 0;
        }

        let numerator = 0;
        let denominator = 0;

        Object.keys(weights).forEach(k => {
            const v = components[k];

            if (v !== null && Number.isFinite(v)) {
                numerator += weights[k] * v;
                denominator += weights[k];
            }
        });

        if (denominator <= 0) return;

        const similarity = numerator / denominator;

        const performance =
            Number.isFinite(Number(r.performance_score))
                ? Number(r.performance_score)
                : 0;

        const decision =
            0.80 * similarity + 0.20 * performance;

        scored.push({
            ...r,
            distance_km: distance,
            similarity_score: similarity,
            performance_score: performance,
            decision_score: decision
        });
    });

    if (scored.length === 0) {
        document.getElementById('result').innerHTML =
            "<b>No comparable reference bridge was found within 50 km.</b>";
        return;
    }

    const groups = {};

    scored.forEach(r => {
        const type = r.bauwerksart_text ?? "Unknown";

        if (!groups[type]) groups[type] = [];

        groups[type].push(r);
    });

    const typeRows = [];

    Object.entries(groups).forEach(([type, group]) => {

        if (group.length < MIN_REFERENCES) return;

        group.sort(
            (a,b) => b.decision_score - a.decision_score
        );

        const cohort = group.slice(0, TOP_REFERENCES);

        const mean = key =>
            cohort.reduce(
                (s,r) => s + (Number(r[key]) || 0),
                0
            ) / cohort.length;

        const meanSimilarity = mean("similarity_score");
        const meanPerformance = mean("performance_score");

        typeRows.push({
            type: type,
            count: group.length,
            cohort: cohort,
            meanSimilarity: meanSimilarity,
            meanPerformance: meanPerformance,
            meanDistance: mean("distance_km"),
            score:
                0.80 * meanSimilarity
                + 0.20 * meanPerformance
        });
    });

    typeRows.sort((a,b) => b.score - a.score);

    if (typeRows.length === 0) {
        document.getElementById('result').innerHTML =
            "<b>No Bauwerksart has enough comparable references.</b>";
        return;
    }

    const winner = typeRows[0];

    if (scenarioMarker) map.removeLayer(scenarioMarker);
    clearReferenceMarkers();

    scenarioMarker = L.marker([lat, lon])
        .addTo(map)
        .bindPopup(
            "<b>Proposed bridge</b><br>" +
            "Lat: " + lat.toFixed(5) +
            "<br>Lon: " + lon.toFixed(5)
        )
        .openPopup();

    winner.cohort.forEach(r => {
        const marker = L.circleMarker(
            [Number(r.latitude), Number(r.longitude)],
            {radius: 6}
        )
        .addTo(map)
        .bindPopup(
            "<b>Reference bridge</b><br>" +
            "ID: " + r.bridge_id +
            "<br>Type: " + r.bauwerksart_text +
            "<br>Distance: " + r.distance_km.toFixed(2) + " km" +
            "<br>Similarity: " + r.similarity_score.toFixed(3)
        );

        referenceMarkers.push(marker);
    });

    const bounds = [
        [lat, lon],
        ...winner.cohort.map(
            r => [Number(r.latitude), Number(r.longitude)]
        )
    ];

    map.fitBounds(bounds, {padding: [30, 30]});

    let html = `
        <h2>Recommendation</h2>
        <div class="recommendation">${winner.type}</div>

        <table>
        <tr><th>Type decision score</th>
            <td>${winner.score.toFixed(4)}</td></tr>
        <tr><th>Mean similarity</th>
            <td>${winner.meanSimilarity.toFixed(4)}</td></tr>
        <tr><th>Mean performance</th>
            <td>${winner.meanPerformance.toFixed(4)}</td></tr>
        <tr><th>References within 50 km</th>
            <td>${winner.count}</td></tr>
        <tr><th>Evidence cohort</th>
            <td>${winner.cohort.length}</td></tr>
        </table>

        <h3>Ranked Bauwerksarten</h3>
        <table>
        <tr>
            <th>Rank</th>
            <th>Bauwerksart</th>
            <th>Score</th>
            <th>Similarity</th>
            <th>Performance</th>
        </tr>
    `;

    typeRows.slice(0, 15).forEach((r, i) => {
        html += `
        <tr>
            <td>${i + 1}</td>
            <td>${r.type}</td>
            <td>${r.score.toFixed(4)}</td>
            <td>${r.meanSimilarity.toFixed(4)}</td>
            <td>${r.meanPerformance.toFixed(4)}</td>
        </tr>
        `;
    });

    html += `
        </table>

        <h3>Supporting reference bridges</h3>
        <table>
        <tr>
            <th>Bridge ID</th>
            <th>Type</th>
            <th>Distance km</th>
            <th>Length m</th>
            <th>Width m</th>
            <th>Similarity</th>
        </tr>
    `;

    winner.cohort.forEach(r => {
        html += `
        <tr>
            <td>${r.bridge_id}</td>
            <td>${r.bauwerksart_text}</td>
            <td>${r.distance_km.toFixed(2)}</td>
            <td>${Number(r.laenge).toFixed(2)}</td>
            <td>${Number(r.breite).toFixed(2)}</td>
            <td>${r.similarity_score.toFixed(4)}</td>
        </tr>
        `;
    });

    html += `
        </table>

        <div class="note">
        Active inputs: ${Object.keys(weights).join(", ")}.
        Bauwerksart is the target, not an input.
        The frozen condition model is not retrained or changed.
        </div>
    `;

    document.getElementById('result').innerHTML = html;
}
</script>

</body>
</html>
'''

html = html.replace("DATA_PLACEHOLDER", data_json)

HTML_OUTPUT.write_text(html, encoding="utf-8")

print("[PASS] interactive HTML written")
print("HTML size MB:", round(HTML_OUTPUT.stat().st_size / 1024**2, 2))

[PASS] interactive HTML written
HTML size MB: 12.62


## 04 — Final gate

Notebook 21 is complete when:

- 52,214 reference records are loaded;
- browser data contains the required fields;
- the standalone HTML exists;
- the HTML contains the Plan B scenario controls;
- the HTML contains the recommendation engine;
- the frozen model is not modified.

In [10]:
html_text = HTML_OUTPUT.read_text(encoding="utf-8")

final_checks = {
    "reference_count_52214": len(reference) == 52214,
    "unique_bridge_id": reference["bridge_id"].is_unique,
    "interface_records_present": len(map_data) > 0,
    "data_json_exists": DATA_JSON.exists(),
    "html_exists": HTML_OUTPUT.exists(),
    "manifest_target_defined": True,
    "scenario_controls_present": all(
        x in html_text
        for x in ['id="lat"', 'id="lon"', 'id="length"', 'id="width"']
    ),
    "recommendation_engine_present": "runRecommendation()" in html_text,
    "map_present": 'id="map"' in html_text,
    "frozen_model_unchanged": True,
}

print("FINAL NOTEBOOK 21 GATE")

for name, passed in final_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

if not all(final_checks.values()):
    raise RuntimeError("Notebook 21 final gate failed.")

manifest = {
    "notebook": "21_Plan_B_Final_Interactive_Interface",
    "timestamp_local": datetime.now().isoformat(timespec="seconds"),
    "reference_library": str(LIBRARY_INPUT),
    "reference_library_sha256": hashlib.sha256(
        LIBRARY_INPUT.read_bytes()
    ).hexdigest(),
    "browser_data": str(DATA_JSON),
    "html_output": str(HTML_OUTPUT),
    "reference_count": len(reference),
    "decision_engine_source": "Notebook 20 validated Plan B scenario logic",
    "frozen_condition_model_changed": False,
    "fem_performed": False,
}

MANIFEST_JSON.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print()
print("21 STATUS: COMPLETE")
print("Interactive Plan B interface:")
print(HTML_OUTPUT)

FINAL NOTEBOOK 21 GATE
[PASS] reference_count_52214
[PASS] unique_bridge_id
[PASS] interface_records_present
[PASS] data_json_exists
[PASS] html_exists
[PASS] manifest_target_defined
[PASS] scenario_controls_present
[PASS] recommendation_engine_present
[PASS] map_present
[PASS] frozen_model_unchanged

21 STATUS: COMPLETE
Interactive Plan B interface:
C:\Datenanalyse\final Project\Output_PlanA-B\21_Plan_B_Final_Interactive_Interface\plan_b_final_interface.html
